In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

In [ ]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import jax.numpy as jnp

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import (
    FINAL_MATERIALS,
    TestSet,
    ResultSet,
    predict_test_scenarios,
    validate_result_set,
    visualize_result_set,
    evaluate_test_scenarios,
    update_pareto_df,
    get_exp_ids_per_material,
)
from rhmag.model_setup import setup_normalizer, setup_dataset

In [ ]:
test_data_per_material = {material_name: TestSet.from_material_name(material_name) for material_name in FINAL_MATERIALS}

#### Quantitative Comparison:

In [ ]:
warmup_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name="ablation-default-f32",
    ),
    test_data_per_material=test_data_per_material,
    force_warmup=True,
)

In [ ]:
one_step_warmup_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-init-zeros-f32",  # exp_name is not fitting, but data is correct
    ),
    test_data_per_material=test_data_per_material,
    force_warmup=False,
)

In [ ]:
no_warmup_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRUZeroStart8",
        exp_name= "ablation-actual-init-zeros-f32",
    ),
    test_data_per_material=test_data_per_material,
    force_warmup=True,
)

In [ ]:
# no_warmup_lin_out_results_df = update_pareto_df(
#     pareto_results_path=None,
#     exp_ids_per_material=get_exp_ids_per_material(
#         model_type="GRULinearOut8",
#         exp_name= "ablation-actual-init-zeros-lin-out-f32",
#     ),
#     test_data_per_material=test_data_per_material,
#     force_warmup=True,
# )

---

In [ ]:
# compare the results

display("warmup:", warmup_results_df.groupby("material").mean(numeric_only=True))
display("one_step:", one_step_warmup_results_df.groupby("material").mean(numeric_only=True))
display("no_warmup:", no_warmup_results_df.groupby("material").mean(numeric_only=True))
display("no_warmup_lin_out_results_df:", no_warmup_lin_out_results_df.groupby("material").mean(numeric_only=True))

In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
# combine into a single df:

warmup_results_df["init_type"] = "replace H (sequence)"
one_step_warmup_results_df["init_type"] = "replace H (one step)"
no_warmup_results_df["init_type"] = "init zeros"
no_warmup_lin_out_results_df["init_type"] = "zero init - linear out"

full_results_df = pd.concat([warmup_results_df, no_warmup_results_df, one_step_warmup_results_df], ignore_index=True)

In [ ]:
import matplotlib as mpl
from matplotlib import rc
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter, LogLocator

rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
def plot_ablation_comparison(full_results_df):
    fig, axs = plt.subplots(1, 4, figsize=(7.167, 7.167 / 3), constrained_layout=True)
    metric_label_map = {
        "sre_avg": r"$\mathrm{SRE}_{\mathrm{avg}}$", 
        "sre_95th": r"$\mathrm{SRE}_{95\mathrm{-th}}$",
        "nere_avg": r"$\mathrm{NERE}_{\mathrm{avg}}$",
        "nere_95th": r"$\mathrm{NERE}_{95\mathrm{-th}}$",
    }
    
    for idx, (ax, metric) in enumerate(zip(axs, ["sre_avg", "sre_95th", "nere_avg", "nere_95th"])):
        sns.stripplot(
            data=full_results_df,
            x="material",
            y=metric,
            hue="init_type",
            palette={"replace H (sequence)": "blue", "replace H (one step)": "red", "init zeros": "orange"},
            legend=True if idx==0 else False,
            ax=ax,
            alpha=0.7,
        )
        ax.set_ylabel(metric_label_map[metric])
        ax.set_yscale('log')
        ax.tick_params(which="both", axis="y", direction="in")
        ax.tick_params(which="both", axis="x", direction="in")
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.3f}'))
        ax.yaxis.set_minor_formatter(StrMethodFormatter('{x:.3f}'))        
        ax.grid(True, which="both", alpha=0.3)

    handles, labels = axs[0].get_legend_handles_labels()

    print(handles, labels)
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3
    )
    axs[0].legend().remove()

    return fig, axs

In [ ]:
fig, axs = plot_ablation_comparison(full_results_df)

# plt.savefig("ablation_study_init_quantitative.png", bbox_inches="tight", dpi=300)
plt.savefig("ablation_study_init_quantitative.pdf", bbox_inches="tight")
plt.show()

In [ ]:
raise

#### Qualitative comparison:

Can you actually see any difference in the qualitative trajectories?

visualize some exemplary trajectories from the data sets

In [ ]:
import numpy as np

In [ ]:
def plot_timeseries(H_past, H_future, B_past, B_future, H_pred, max_n_length=None, figsize=None):
    H_full_true = jnp.concatenate([H_past, H_future])
    B_full_true = jnp.concatenate([B_past, B_future])
    H_full_pred = jnp.concatenate([H_past, H_pred])
    if max_n_length is not None:
        H_full_true = H_full_true[:max_n_length]
        B_full_true = B_full_true[:max_n_length]
        H_full_pred = H_full_pred[:max_n_length]

    tau = 1 / (16)
    t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])
    
    fig, axs = plt.subplots(1,1, figsize=figsize)
    axs.plot(
        t,
        H_full_true,
        color="tab:blue",
        label="$H$",
    )
    axs.plot(
        t,
        H_full_pred,
        color="tab:orange",
        label="$\hat{H}$",
        linestyle="dashed",
    )
    axs.set_ylabel("$H \mathrm{\; in \; A/m}$")
    axs.set_xlabel("$t \mathrm{\; in \; \\upmu s}$")
    axs.grid(True, alpha=0.3)
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    fig.tight_layout()
    return fig, axs


def plot_BH_curve_with_loss(H_past, H_future, B_past, B_future, H_pred, max_n_length=None, figsize=None):

    H_full_true = jnp.concatenate([H_past, H_future])
    B_full_true = jnp.concatenate([B_past, B_future])
    H_full_pred = jnp.concatenate([H_past, H_pred])
    if max_n_length is not None:
        H_full_true = H_full_true[:max_n_length]
        B_full_true = B_full_true[:max_n_length]
        H_full_pred = H_full_pred[:max_n_length]
    
    fig, axs = plt.subplots(1,1, figsize=figsize)#(7.167/2,7.167/2))
    axs.plot(
        H_full_true,
        B_full_true,
        color="tab:blue",
        label="$BH$",
    )
    axs.plot(
        H_full_pred,
        B_full_true,
        color="tab:orange",
        label="$B\\hat{H}$",
        linestyle="dashed",
    )

    axs.set_xlabel("$H \mathrm{\; in \; A/m}$")
    axs.set_ylabel("$B \mathrm{\; in \; T}$")
    axs.tick_params(which="major", axis="y", direction="in")
    axs.tick_params(which="both", axis="x", direction="in")
    axs.legend()
    axs.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, axs

In [ ]:
material_name = "A"
model_idx = 0

(training_set, eval_set, test_set) = setup_dataset(
    material_name=material_name,
    subsampling_freq=1,
    use_all_data=False,
)

#test_set = test_data_per_material[material_name]
model_no_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-init-zeros-f32",
    )[material_name][model_idx]
)

model_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-default-f32",
    )[material_name][model_idx]
)

past_size = 100
frequency = 80_000

H_future=test_set.at_frequency(frequency).H[:, past_size:]
B_past=test_set.at_frequency(frequency).B[:, :past_size]
H_past=test_set.at_frequency(frequency).H[:, :past_size]
B_future=test_set.at_frequency(frequency).B[:, past_size:]
T=test_set.at_frequency(frequency).T

H_pred_no_warmup = model_no_warmup(
    B_past=B_past[:, -1:],
    H_past=H_past[:, -1:],
    B_future=B_future,
    T=T,
    warmup=False,
)

H_pred_warmup = model_warmup(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

idx = 0

fig, axs = plot_timeseries(
    H_past[idx],
    H_future[idx],
    B_past[idx],
    B_future[idx],
    H_pred_warmup[idx],
    max_n_length=250,
    figsize=(7.167/2,7.167/3),
)

H_full_true = jnp.concatenate([H_past[idx], H_future[idx]])
H_full_pred = jnp.concatenate([H_past[idx], H_pred_no_warmup[idx]])
H_full_true = H_full_true[:250]
H_full_pred = H_full_pred[:250]

tau = 1 / (16)
t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])

axs.plot(t, H_full_pred, color="tab:purple", linestyle="dashed", label="$\hat{H}_\mathrm{init \, zeros}$",)

t0 = 0
t1 = (H_past[idx].shape[0] -1) * tau
axs.vlines(
    x=[t1],
    ymin = axs.get_ylim()[0], # - (0.06 * axs.get_ylim()[0]),
    ymax=axs.get_ylim()[1],
    color="k",
    #linestyles="dashed",
    linewidth=1,
)
axs.text(t1 - 3.3, -20, '$\\textbf{warmup}$', fontsize=10, color="k", verticalalignment="center")
axs.text(t1 + 0.25, -20, '$\\textbf{prediction}$', fontsize=10, color="k", verticalalignment="center")

axs.set_xlim(t[0], t[-1])
axs.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

plt.savefig("ablation_init_qualitative_1_one_step.pdf", bbox_inches="tight")

plt.show()

# fig, axs = plot_BH_curve_with_loss(
#     H_past[idx],
#     H_future[idx],
#     B_past[idx],
#     B_future[idx],
#     H_pred_MSE[idx],
#     max_n_length=None,
#     figsize=(7.167/2,7.167/3),
# )
# plt.show()

In [ ]:
material_name = "D"
model_idx = 0

(training_set, eval_set, test_set) = setup_dataset(
    material_name=material_name,
    subsampling_freq=1,
    use_all_data=False,
)

#test_set = test_data_per_material[material_name]
model_no_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-init-zeros-f32",
    )[material_name][model_idx]
)

model_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-default-f32",
    )[material_name][model_idx]
)

past_size = 100
frequency = 320_000

H_future=test_set.at_frequency(frequency).H[:, past_size:]
B_past=test_set.at_frequency(frequency).B[:, :past_size]
H_past=test_set.at_frequency(frequency).H[:, :past_size]
B_future=test_set.at_frequency(frequency).B[:, past_size:]
T=test_set.at_frequency(frequency).T

H_pred_no_warmup = model_no_warmup(
    B_past=B_past[:, -1:],
    H_past=H_past[:, -1:],
    B_future=B_future,
    T=T,
    warmup=False,
)

H_pred_warmup = model_warmup(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

idx = 0

fig, axs = plot_timeseries(
    H_past[idx],
    H_future[idx],
    B_past[idx],
    B_future[idx],
    H_pred_warmup[idx],
    max_n_length=250,
    figsize=(7.167/2,7.167/3),
)

H_full_true = jnp.concatenate([H_past[idx], H_future[idx]])
H_full_pred = jnp.concatenate([H_past[idx], H_pred_no_warmup[idx]])
H_full_true = H_full_true[:250]
H_full_pred = H_full_pred[:250]

tau = 1 / (16)
t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])

axs.plot(t, H_full_pred, color="tab:purple", linestyle="dashed", label="$\hat{H}_\mathrm{init \, zeros}$",)

t0 = 0
t1 = (H_past[idx].shape[0] -1) * tau
axs.vlines(
    x=[t1],
    ymin = axs.get_ylim()[0], # - (0.06 * axs.get_ylim()[0]),
    ymax=axs.get_ylim()[1],
    color="k",
    #linestyles="dashed",
    linewidth=1,
)
axs.text(t1 - 3.3, -20, '$\\textbf{warmup}$', fontsize=10, color="k", verticalalignment="center")
axs.text(t1 + 0.25, -20, '$\\textbf{prediction}$', fontsize=10, color="k", verticalalignment="center")

axs.set_xlim(t[0], t[-1])
axs.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

plt.savefig("ablation_init_qualitative_2_one_step.pdf", bbox_inches="tight")

plt.show()

# fig, axs = plot_BH_curve_with_loss(
#     H_past[idx],
#     H_future[idx],
#     B_past[idx],
#     B_future[idx],
#     H_pred_MSE[idx],
#     max_n_length=None,
#     figsize=(7.167/2,7.167/3),
# )
# plt.show()

In [ ]:
material_name = "A"
model_idx = 0

(training_set, eval_set, test_set) = setup_dataset(
    material_name=material_name,
    subsampling_freq=1,
    use_all_data=False,
)

#test_set = test_data_per_material[material_name]
model_no_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRUZeroStart8",
        exp_name= "ablation-actual-init-zeros-f32",
    )[material_name][model_idx]
)

model_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-default-f32",
    )[material_name][model_idx]
)

past_size = 100
frequency = 80_000

H_future=test_set.at_frequency(frequency).H[:, past_size:]
B_past=test_set.at_frequency(frequency).B[:, :past_size]
H_past=test_set.at_frequency(frequency).H[:, :past_size]
B_future=test_set.at_frequency(frequency).B[:, past_size:]
T=test_set.at_frequency(frequency).T

H_pred_no_warmup = model_no_warmup(
    B_past=B_past[:, -1:],
    H_past=H_past[:, -1:],
    B_future=B_future,
    T=T,
)

H_pred_warmup = model_warmup(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

idx = 0

fig, axs = plot_timeseries(
    H_past[idx],
    H_future[idx],
    B_past[idx],
    B_future[idx],
    H_pred_warmup[idx],
    max_n_length=250,
    figsize=(7.167/2,7.167/3),
)

H_full_true = jnp.concatenate([H_past[idx], H_future[idx]])
H_full_pred = jnp.concatenate([H_past[idx], H_pred_no_warmup[idx]])
H_full_true = H_full_true[:250]
H_full_pred = H_full_pred[:250]

tau = 1 / (16)
t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])

axs.plot(t, H_full_pred, color="tab:purple", linestyle="dashed", label="$\hat{H}_\mathrm{init \, zeros}$",)

t0 = 0
t1 = (H_past[idx].shape[0] -1) * tau
axs.vlines(
    x=[t1],
    ymin = axs.get_ylim()[0], # - (0.06 * axs.get_ylim()[0]),
    ymax=axs.get_ylim()[1],
    color="k",
    #linestyles="dashed",
    linewidth=1,
)
axs.text(t1 - 3.3, -20, '$\\textbf{warmup}$', fontsize=10, color="k", verticalalignment="center")
axs.text(t1 + 0.25, -20, '$\\textbf{prediction}$', fontsize=10, color="k", verticalalignment="center")

axs.set_xlim(t[0], t[-1])
axs.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

plt.savefig("ablation_init_qualitative_1.pdf", bbox_inches="tight")

plt.show()

# fig, axs = plot_BH_curve_with_loss(
#     H_past[idx],
#     H_future[idx],
#     B_past[idx],
#     B_future[idx],
#     H_pred_MSE[idx],
#     max_n_length=None,
#     figsize=(7.167/2,7.167/3),
# )
# plt.show()

In [ ]:
material_name = "D"
model_idx = 0

(training_set, eval_set, test_set) = setup_dataset(
    material_name=material_name,
    subsampling_freq=1,
    use_all_data=False,
)

#test_set = test_data_per_material[material_name]
model_no_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRUZeroStart8",
        exp_name= "ablation-actual-init-zeros-f32",
    )[material_name][model_idx]
)

model_warmup = reconstruct_model_from_file(
    filename=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name= "ablation-default-f32",
    )[material_name][model_idx]
)

past_size = 100
frequency = 500_000

H_future=test_set.at_frequency(frequency).H[:, past_size:]
B_past=test_set.at_frequency(frequency).B[:, :past_size]
H_past=test_set.at_frequency(frequency).H[:, :past_size]
B_future=test_set.at_frequency(frequency).B[:, past_size:]
T=test_set.at_frequency(frequency).T

H_pred_no_warmup = model_no_warmup(
    B_past=B_past[:, -1:],
    H_past=H_past[:, -1:],
    B_future=B_future,
    T=T,
)

H_pred_warmup = model_warmup(
    B_past=B_past,
    H_past=H_past,
    B_future=B_future,
    T=T,
)

idx = 0

fig, axs = plot_timeseries(
    H_past[idx],
    H_future[idx],
    B_past[idx],
    B_future[idx],
    H_pred_warmup[idx],
    max_n_length=250,
    figsize=(7.167/2,7.167/3),
)

H_full_true = jnp.concatenate([H_past[idx], H_future[idx]])
H_full_pred = jnp.concatenate([H_past[idx], H_pred_no_warmup[idx]])
H_full_true = H_full_true[:250]
H_full_pred = H_full_pred[:250]

tau = 1 / (16)
t = np.linspace(0, (H_full_true.shape[0] -1) * tau, H_full_true.shape[0])

axs.plot(t, H_full_pred, color="tab:purple", linestyle="dashed", label="$\hat{H}_\mathrm{init \, zeros}$",)

t0 = 0
t1 = (H_past[idx].shape[0] -1) * tau
axs.vlines(
    x=[t1],
    ymin = axs.get_ylim()[0], # - (0.06 * axs.get_ylim()[0]),
    ymax=axs.get_ylim()[1],
    color="k",
    #linestyles="dashed",
    linewidth=1,
)
axs.text(t1 - 3.3, -25, '$\\textbf{warmup}$', fontsize=10, color="k", verticalalignment="center")
axs.text(t1 + 0.25, -25, '$\\textbf{prediction}$', fontsize=10, color="k", verticalalignment="center")

axs.set_xlim(t[0], t[-1])
axs.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3)

plt.savefig("ablation_init_qualitative_2.pdf", bbox_inches="tight")

plt.show()

# fig, axs = plot_BH_curve_with_loss(
#     H_past[idx],
#     H_future[idx],
#     B_past[idx],
#     B_future[idx],
#     H_pred_MSE[idx],
#     max_n_length=None,
#     figsize=(7.167/2,7.167/3),
# )
# plt.show()

In [ ]:
H_past.shape[0]

- Maybe it makes sense to show mean +- std of the models for the simulation trajectory

In [ ]:
# plot the second model's prediction into the same figure:
raise NotImplementedError


for scenario in test_set.scenarios:
    H_pred = model(
        B_past=scenario.B_past,
        H_past=scenario.H_past,
        B_future=scenario.B_future,
        T=jnp.squeeze(scenario.T),
    )

    B_future = scenario.B_future
    H_future = scenario.H_future
    B_past = scenario.B_past

    start_idx = 0
    n_plots = 5

    for start_idx in np.arange(0, H_pred.shape[0], n_plots):
    
        fig, axs = plt.subplots(3, n_plots, figsize=(12,7))
        for idx in range(n_plots):
            axs[0, idx].plot(B_future[start_idx+idx])
            axs[1, idx].plot(H_future[start_idx+idx])
            axs[1, idx].plot(H_pred[start_idx+idx])
            axs[1, idx].plot(H_future[start_idx+idx] - H_pred[start_idx+idx], color="tab:red", linestyle="--")
        
            axs[2, idx].plot(B_future[start_idx+idx], H_future[start_idx+idx])
            axs[2, idx].plot(B_future[start_idx+idx], H_pred[start_idx+idx])
        
            axs[0, idx].grid(True, alpha=0.3)
            axs[1, idx].grid(True, alpha=0.3)
            axs[2, idx].grid(True, alpha=0.3)
        
            axs[0, idx].set_ylabel("B")
            axs[0, idx].set_xlabel("k")
            axs[1, idx].set_ylabel("H")
            axs[1, idx].set_xlabel("k")
            axs[2, idx].set_ylabel("H")
            axs[2, idx].set_xlabel("B")
        
        fig.tight_layout(pad=-0.2)
        plt.show()